# 11 · Leapfrog exchange with OMF (experimental)

`subsurfix.omf_io` writes and reads OMF (Open Mining Format) v1 files, the open format Leapfrog imports and exports.
Export a property grid with uncertainty bands, well trajectories with logs, and horizon surfaces; import Leapfrog
surfaces or block models back as numpy arrays to use as a structural framework. Synthetic data keeps this notebook
self-contained. **Experimental:** the API may change.

In [1]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from subsurfix import omf_io

warnings.filterwarnings("ignore", message=".*omf_io is experimental")
warnings.filterwarnings("ignore", category=DeprecationWarning)
OUT = Path(os.environ.get("SX_EXAMPLE_OUT", "outputs")).resolve()
OUT.mkdir(parents=True, exist_ok=True)

## Build a small grid, two wells and a horizon

In [2]:
nx, ny, nz = 20, 15, 6
dx, dy, dz = 100.0, 100.0, 10.0
origin = (0.0, 0.0, -1600.0)
i, j, k = np.meshgrid(np.arange(nx), np.arange(ny), np.arange(nz), indexing="ij")
i, j, k = (a.ravel(order="F") for a in (i, j, k))  # i fastest
poro = 0.12 + 0.08 * np.sin(i / 4.0) * np.cos(j / 5.0) + 0.01 * k
grid = {
    "origin": origin, "cell_size": (dx, dy, dz), "dims": (nx, ny, nz),
    "properties": {"PORO": poro, "PORO_P10": poro - 0.03, "PORO_P90": poro + 0.03,
                   "PERM_mD": 10 ** (12 * poro - 0.5)},
}
z = np.linspace(-1600, -1540, 31)
wells = [
    {"name": "W-A", "xyz": np.column_stack([np.full(31, 550.0), np.full(31, 420.0), z[::-1]]),
     "logs": {"GR": 60 + 30 * np.sin(z / 9.0)}},
    {"name": "W-B", "xyz": np.column_stack([np.linspace(1400, 1500, 31), np.full(31, 1100.0), z[::-1]]),
     "logs": {"GR": 75 + 20 * np.cos(z / 7.0)}},
]
x, y = np.linspace(0, nx * dx, 21), np.linspace(0, ny * dy, 16)
top = -1560.0 - 0.01 * x[None, :] + 0.005 * y[:, None]
surfaces = {"Top_Reservoir": {"x": x, "y": y, "z": top}}

## Export to OMF (open this file in Leapfrog via *Import → OMF*)

In [3]:
path = omf_io.export_omf(OUT / "subsurfix_leapfrog.omf", grid=grid, wells=wells, surfaces=surfaces)
print(path.name, f"{path.stat().st_size / 1024:.0f} KB")

subsurfix_leapfrog.omf 60 KB


## Import it back, as you would a Leapfrog export

In [4]:
m = omf_io.import_omf(path)
bm, srf = m["block_models"][0], m["surfaces"][0]
print("block model dims", bm["dims"], "properties", sorted(bm["properties"]))
print("wells", [w["name"] for w in m["wells"]], "surfaces", [s["name"] for s in m["surfaces"]])
assert np.allclose(bm["properties"]["PORO"], poro) and np.allclose(srf["z"], top)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
p = bm["properties"]["PORO"].reshape(bm["dims"], order="F")[:, :, 0].T
ax[0].imshow(p, origin="lower", extent=(0, nx * dx, 0, ny * dy)); ax[0].set_title("Re-imported PORO, layer 1")
for w in m["wells"]:
    ax[0].plot(*w["xyz"][:, :2].T, "wo", ms=3)
c = ax[1].contourf(srf["x"], srf["y"], srf["z"], 12); fig.colorbar(c, ax=ax[1], label="z (m)")
ax[1].set_title("Re-imported Top_Reservoir")
fig.tight_layout()

block model dims (20, 15, 6) properties ['PERM_mD', 'PORO', 'PORO_P10', 'PORO_P90']
wells ['W-A', 'W-B'] surfaces ['Top_Reservoir']


Imported surfaces (`x`, `y`, `z` with shape (ny, nx)) can feed grid building as the structural framework.
Caveats: OMF v1 has no corner-point grid (a `CornerPointGrid` is exported as cell-centre points), rotated
Leapfrog grids keep their axes in `axes`, and coordinates are passed through unchanged (no CRS conversion).